## Part A — The Classification Setup


## TASK 01 · Encode, split, and standardize

In [ ]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Load your Titanic CSV file
df = pd.read_csv("titanic.csv")

# 2. Handle missing values
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Fare"] = df["Fare"].fillna(df["Fare"].median())

# 3. Convert Sex into binary values
# Female = 1, Male = 0
df["Sex"] = (df["Sex"] == "female").astype(int)

# 4. Separate features (X) and target (y)
X = df.drop(columns=["Survived"])
y = df["Survived"].values

# 5. Split data into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 6. Standardize features using training data only
scaler = StandardScaler()
scaler.fit(X_train)

X_tr = scaler.transform(X_train)
X_te = scaler.transform(X_test)

# 7. Check the results
print("Original dataset shape:", df.shape)
print("Training features shape:", X_tr.shape)
print("Testing features shape:", X_te.shape)

print("\nTraining target shape:", y_train.shape)
print("Testing target shape:", y_test.shape)

print("\nFirst 5 scaled training rows:")
print(X_tr[:5])


1\. Why is stratified splitting essential in classification?

Stratified splitting maintains approximately the same proportion of each class in training and test sets. For Titanic, it helps preserve the ratio of survivors (`1`) and non-survivors (`0`), making evaluation more reliable.

2\. What exact leakage occurs if test data is scaled together with training data?

If we fit `StandardScaler` on the entire dataset, the test data influences the calculated mean and standard deviation. This is data leakage because information from the test set enters the preprocessing process, potentially making evaluation less reliable.

## Part B — Build and Prove (The Core Deliverable)

## TASK 02 - The Sigmoid function and Log-Loss

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# Vectorized sigmoid function
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Generate values from -10 to +10
z = np.linspace(-10, 10, 200)
probabilities = sigmoid(z)

# Plot sigmoid curve
plt.figure(figsize=(8, 5))
plt.plot(z, probabilities, label="Sigmoid")
plt.axhline(y=0.5, color="red", linestyle="--", label="Threshold = 0.5")
plt.axvline(x=0, color="gray", linestyle=":")
plt.xlabel("z")
plt.ylabel("Probability")
plt.title("Sigmoid Function")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## TASK 03 - Method 1 — Logistic Regression from scratch

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Load Titanic dataset
df = pd.read_csv("titanic.csv")

# 2. Handle missing values
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Fare"] = df["Fare"].fillna(df["Fare"].median())

# 3. Encode Sex: female = 1, male = 0
df["Sex"] = (df["Sex"] == "female").astype(int)

# 4. Separate features and target
X = df.drop(columns=["Survived"])
y = df["Survived"].to_numpy()

# 5. Split into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 6. Scale features using training data only
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# 7. Logistic Regression from scratch
class LogisticRegressionScratch:
    def __init__(self, learning_rate=0.01, iterations=2000):
        self.lr = learning_rate
        self.iterations = iterations
        self.w = None
        self.b = 0
        self.loss_history = []

    def sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        m, n = X.shape
        self.w = np.zeros(n)
        self.b = 0

        for _ in range(self.iterations):
            # Calculate predicted probabilities
            p = self.sigmoid(X @ self.w + self.b)

            # Calculate binary cross-entropy loss
            eps = 1e-15
            p_safe = np.clip(p, eps, 1 - eps)
            loss = -np.mean(
                y * np.log(p_safe) +
                (1 - y) * np.log(1 - p_safe)
            )
            self.loss_history.append(loss)

            # Calculate gradients
            dw = (X.T @ (p - y)) / m
            db = np.sum(p - y) / m

            # Update weights and bias
            self.w -= self.lr * dw
            self.b -= self.lr * db

        return self

    def predict_proba(self, X):
        return self.sigmoid(X @ self.w + self.b)

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)


# 8. Train the model
model = LogisticRegressionScratch(
    learning_rate=0.01,
    iterations=2000
)
model.fit(X_train, y_train)

# 9. Plot training loss
plt.figure(figsize=(8, 5))
plt.plot(model.loss_history)
plt.xlabel("Iteration")
plt.ylabel("Binary Cross-Entropy Loss")
plt.title("Logistic Regression: Training Loss")
plt.grid(True)
plt.tight_layout()
plt.show()

# 10. Check predictions and loss
print("Initial loss:", model.loss_history[0])
print("Final loss:", model.loss_history[-1])

predictions = model.predict(X_test)
print("First 10 predictions:", predictions[:10])


## TASK 04 · Method 2 — Scikit-Learn parity proof

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# 1. Load Titanic dataset
df = pd.read_csv("titanic.csv")

# 2. Handle missing values
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Fare"] = df["Fare"].fillna(df["Fare"].median())

# 3. Encode Sex: female = 1, male = 0
df["Sex"] = (df["Sex"] == "female").astype(int)

# 4. Separate features and target
X = df.drop(columns=["Survived"])
y = df["Survived"].to_numpy()

# 5. Split into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 6. Scale features using training data only
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# 5. From-scratch logistic regression
# This implementation uses gradient descent with backtracking
class LogisticRegressionScratch:
    def __init__(self, max_iter=5000, tol=1e-12):
        self.max_iter = max_iter
        self.tol = tol

    def sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))

    def loss(self, X, y, w, b):
        z = X @ w + b
        return np.mean(np.logaddexp(0, z) - y * z)

    def fit(self, X, y):
        y = np.asarray(y, dtype=float)
        m, n = X.shape
        self.w = np.zeros(n)
        self.b = 0.0
        theta = np.zeros(n + 1)
        A = np.column_stack((X, np.ones(m)))
        current_loss = self.loss(X, y, self.w, self.b)

        for _ in range(self.max_iter):
            p = self.sigmoid(X @ self.w + self.b)
            grad = np.concatenate((
                X.T @ (p - y) / m,
                [np.mean(p - y)]
            ))

            if np.linalg.norm(grad) < self.tol:
                break

            step = 1.0
            old_theta = np.concatenate((self.w, [self.b]))

            while step > 1e-12:
                candidate = old_theta - step * grad
                new_loss = self.loss(
                    X, y, candidate[:-1], candidate[-1]
                )
                if new_loss <= current_loss:
                    break
                step *= 0.5

            if step <= 1e-12:
                break

            theta = candidate
            self.w = theta[:-1]
            self.b = theta[-1]
            current_loss = new_loss

        return self

    def predict(self, X):
        return self.sigmoid(X @ self.w + self.b)

scratch = LogisticRegressionScratch().fit(X_tr, y_train)

# 6. Scikit-learn model
sk_model = LogisticRegression(
    penalty=None, solver="lbfgs", max_iter=10000, tol=1e-12
)
sk_model.fit(X_tr, y_train)

# 7. Compare intercept and feature weights
parity_df = pd.DataFrame({
    "Parameter": ["Intercept"] + features,
    "Scratch": np.r_[scratch.b, scratch.w],
    "Sklearn": np.r_[sk_model.intercept_[0], sk_model.coef_[0]]})
parity_df["Absolute Difference"] = (
    parity_df["Scratch"] - parity_df["Sklearn"]
).abs()

print("INTERCEPT AND FEATURE WEIGHTS")
print(parity_df.to_string(index=False))

params_match = np.allclose(
    parity_df["Scratch"], parity_df["Sklearn"], atol=1e-3
)
print("\nAll parameters match within 1e-3:", params_match)

# 8. Compare predictions on the test set
scratch_pred = (scratch.predict(X_te) >= 0.5).astype(int)
sklearn_pred = sk_model.predict(X_te)

print("Test predictions match:",
    np.array_equal(scratch_pred, sklearn_pred))
print("Matching test predictions:",
    f"{np.sum(scratch_pred == sklearn_pred)}/{len(y_test)}")

## Part C — Measure, Tune, and Interpret

## TASK 05 · Confusion Matrix and Classification Metrics

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# 1. Predict on the test set
y_pred = sk_model.predict(X_te)

# 2. Calculate confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix Values:")
print("TN (True Negative):", tn)
print("FP (False Positive):", fp)
print("FN (False Negative):", fn)
print("TP (True Positive):", tp)

# 3. Plot annotated heatmap
plt.figure(figsize=(6, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Predicted Not Survived", "Predicted Survived"],
    yticklabels=["Actual Not Survived", "Actual Survived"]
)
plt.title("Titanic Test Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.tight_layout()
plt.show()

# 4. Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("\nClassification Metrics:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-Score : {f1:.4f}")

##  TASK 06 · Decision Threshold Tuning and the ROC-AUC Curve

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    precision_score,
    recall_score,
    roc_curve,
    roc_auc_score
)

# 1. Extract predicted probabilities for survival (class 1)
probs = sk_model.predict_proba(X_te)[:, 1]

# 2. Compare Precision and Recall at different thresholds
thresholds = [0.2, 0.5, 0.8]
results = []

for threshold in thresholds:
    y_pred = (probs >= threshold).astype(int)

    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)

    results.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall
    })

results_df = pd.DataFrame(results)

print("PRECISION-RECALL COMPARISON")
print(results_df.round(4).to_string(index=False))

# 3. Calculate ROC curve and ROC-AUC
fpr, tpr, roc_thresholds = roc_curve(y_test, probs)
auc_score = roc_auc_score(y_test, probs)

# 4. Plot ROC curve
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"Logistic Regression (AUC = {auc_score:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random Classifier")

plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate (TPR / Recall)")
plt.title("ROC Curve - Titanic Survival Prediction")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nROC-AUC Score: {auc_score:.4f}")

## TASK 07 · Odds Ratios and Plain-English Model Interpretation

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Feature names from your Titanic CSV
features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare"]

# Get weights and calculate Odds Ratios
weights = sk_model.coef_[0]
odds_ratios = np.exp(weights)

# Create a results table
results = pd.DataFrame({
    "Feature": features,
    "Weight": weights,
    "Odds_Ratio": odds_ratios
})

print("Feature Weights and Odds Ratios:")
print(results.round(4).to_string(index=False))

# Orange for positive weights, blue for negative weights
colors = ["orange" if weight > 0 else "blue" for weight in weights]

# Horizontal bar chart
plt.figure(figsize=(9, 5))
plt.barh(features, weights, color=colors)
plt.axvline(x=0, color="black", linewidth=0.8)
plt.xlabel("Feature Weight (Coefficient)")
plt.ylabel("Features")
plt.title("Titanic Survival: Feature Weights")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()
